<a href="https://colab.research.google.com/github/Halidh-Ahamed/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import duckdb
from google.colab import userdata

In [18]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [19]:
con = duckdb.connect()

In [20]:
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [21]:
rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected!")

Connected!


In [22]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [23]:
feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,

    client_has_gsc,
    client_has_ga4,

    gsc_data_available

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')

WHERE month='2026-03'
AND gsc_data_available IS TRUE

LIMIT 20
""").df()

feature_df

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True
5,2026-03-01,content_36c36abc7650d7af,239,1,1756,True,False,True
6,2026-03-01,content_a7da352b73b02668,191,0,1496,True,False,True
7,2026-03-01,content_05434271b257bb68,55,0,180,True,False,True
8,2026-03-01,content_d056587ff7faca0c,77,0,434,True,False,True
9,2026-03-01,content_bfd1e41c2af250c8,2,0,9,True,False,True


In [24]:
feature_df["avg_position"] = (
    feature_df["gsc_sum_position"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)
feature_df.head()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available,avg_position,ctr
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True,3.350000,0.000
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True,0.000000,0.000
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True,4.928000,0.008
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True,4.000000,0.000
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True,2.272727,0.000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [25]:
feature_df["avg_position"] = feature_df["avg_position"].fillna(0)
feature_df["ctr"] = feature_df["ctr"].fillna(0)

feature_df.head()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available,avg_position,ctr
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True,3.350000,0.000
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True,0.000000,0.000
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True,4.928000,0.008
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True,4.000000,0.000
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True,2.272727,0.000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [26]:
feature_df.columns

Index(['report_date', 'content_hash_id', 'gsc_impressions', 'gsc_clicks',
       'gsc_sum_position', 'client_has_gsc', 'client_has_ga4',
       'gsc_data_available', 'avg_position', 'ctr'],
      dtype='object')

## Leakage Hunt

I reviewed every feature used in the feature vector to check for possible data leakage.

The selected numerical features (`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `avg_position`, and `ctr`) are calculated from historical observations and are available before the prediction is made.

The boolean fields (`client_has_gsc`, `client_has_ga4`, and `gsc_data_available`) describe the data collection setup and are also known before prediction.

The `content_hash_id` column was excluded from model training because it is a unique identifier. Using it could cause the model to memorize individual pages instead of learning patterns that generalize to unseen content.

No feature in the final feature vector directly contains future information or the prediction label, so no data leakage was observed.


In [27]:
feature_df.dtypes

,0
report_date,datetime64[us]
content_hash_id,object
gsc_impressions,int64
gsc_clicks,int64
gsc_sum_position,int64
client_has_gsc,bool
client_has_ga4,bool
gsc_data_available,bool
avg_position,float64
ctr,float64


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## What I Excluded and Why

- **content_hash_id** – Excluded because it is a unique identifier. Using it could cause the model to memorize individual pages instead of learning general patterns.

- **report_date** – Excluded because the raw date is not directly useful as a predictive feature. If needed, meaningful time-based features can be engineered from it instead.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.